# AI Code Review Assistant — Evaluation Analysis

This notebook loads evaluation results from the AI Code Review Assistant benchmark,
computes metrics, and generates visualizations to analyze the model's detection capabilities.

## Setup

Make sure you've run the evaluation first:
```bash
python -m evaluation.evaluate --mode security
python -m evaluation.metrics evaluation/results/run_<timestamp>.json
```

In [ ]:
import sys
from collections import Counter
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path('.').resolve()))

# Try importing visualization libraries
try:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.rcParams['figure.figsize'] = (12, 6)
    matplotlib.rcParams['font.size'] = 12
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print('matplotlib not installed — tables will be shown, charts skipped')
    print('Install with: pip install matplotlib')

from evaluation.metrics import compute_metrics, generate_markdown_report, load_run

## 1. Load Results

Load the most recent evaluation run (or specify a path).

In [ ]:
# Auto-detect latest run file
results_dir = Path('evaluation/results')
run_files = sorted(results_dir.glob('run_*.json'))
run_files = [f for f in run_files if '_metrics' not in f.name]

if not run_files:
    raise FileNotFoundError(
        'No run files found. Run the evaluation first:\n'
        '  python -m evaluation.evaluate --mode security'
    )

latest = run_files[-1]
print(f'Loading: {latest}')

run_data = load_run(str(latest))
metrics = compute_metrics(run_data)

print(f"Run ID: {metrics['run_id']}")
print(f"Model: {metrics['model']}")
print(f"Mode: {metrics['mode']}")
print(f"Snippets: {metrics['dataset_size']['vulnerable']} vulnerable + {metrics['dataset_size']['safe']} safe")

## 2. Overall Precision / Recall / F1 Summary

In [ ]:
print('=' * 50)
print('OVERALL METRICS')
print('=' * 50)
print(f"Detection Rate (Recall): {metrics['detection_rate_recall']:.1%}")
print(f"Precision:               {metrics['precision']:.1%}")
print(f"F1 Score:                {metrics['f1_score']:.1%}")
print(f"False Positive Rate:     {metrics['false_positive_rate']:.1%}")
print(f"Avg Line Accuracy:       {metrics['avg_line_accuracy']:.1%}")
print()
c = metrics['classification']
print('Classification Matrix:')
print(f"  TP={c['true_positives']}  FN={c['false_negatives']}")
print(f"  FP={c['false_positives']}  TN={c['true_negatives']}")

## 3. Detection Rate by Vulnerability Category

Which vulnerability types does the model find best/worst?

In [ ]:
cats = metrics['category_breakdown']
sorted_cats = sorted(cats.items(), key=lambda x: x[1]['recall'], reverse=True)

print(f"{'Category':<30} {'Total':>6} {'Detected':>9} {'Recall':>8}")
print('-' * 55)
for cat, stats in sorted_cats:
    print(f"{cat:<30} {stats['total']:>6} {stats['detected']:>9} {stats['recall']:>7.0%}")

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(12, 6))
    categories = [c[0] for c in sorted_cats]
    recalls = [c[1]['recall'] for c in sorted_cats]
    colors = ['#2ecc71' if r >= 0.8 else '#f39c12' if r >= 0.5 else '#e74c3c' for r in recalls]
    bars = ax.barh(categories, recalls, color=colors)
    ax.set_xlabel('Recall')
    ax.set_title('Detection Rate by Vulnerability Category')
    ax.set_xlim(0, 1.05)
    for bar, val in zip(bars, recalls, strict=False):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:.0%}', va='center', fontsize=10)
    plt.tight_layout()
    plt.show()

## 4. Confidence Calibration

Are high-confidence findings actually correct? A well-calibrated model should show
accuracy increasing with confidence.

In [ ]:
cal = metrics['confidence_calibration']

print(f"{'Bin':<12} {'Range':<10} {'Total':>6} {'Correct':>8} {'Accuracy':>9}")
print('-' * 48)
for label, data in cal.items():
    print(f"{label:<12} {data['range']:<10} {data['total']:>6} {data['correct']:>8} {data['accuracy']:>8.0%}")

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 6))
    labels = list(cal.keys())
    accuracies = [cal[label]['accuracy'] for label in labels]
    totals = [cal[label]['total'] for label in labels]
    
    # Perfect calibration line
    ideal_midpoints = [0.15, 0.45, 0.7, 0.9]
    ax.plot(ideal_midpoints, ideal_midpoints, 'k--', alpha=0.5, label='Perfect calibration')
    ax.bar(range(len(labels)), accuracies, tick_label=labels, alpha=0.7, color='#3498db')
    ax.set_ylabel('Actual Accuracy')
    ax.set_xlabel('Confidence Bin')
    ax.set_title('Confidence Calibration')
    ax.set_ylim(0, 1.05)
    
    # Annotate with counts
    for i, (acc, tot) in enumerate(zip(accuracies, totals, strict=False)):
        ax.text(i, acc + 0.03, f'n={tot}', ha='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()

## 5. Latency Distribution

In [ ]:
lat = metrics['latency']
print(f"Avg Summary Latency: {lat['avg_summary_ms']:.0f} ms")
print(f"Avg Inline Latency:  {lat['avg_inline_ms']:.0f} ms")
print(f"Avg Total Latency:   {lat['avg_total_ms']:.0f} ms")
print(f"Min Total:           {lat['min_total_ms']:.0f} ms")
print(f"Max Total:           {lat['max_total_ms']:.0f} ms")

if HAS_MPL:
    results = run_data['results']
    total_latencies = []
    for r in results:
        s = r.get('summary', {}).get('latency_ms', 0)
        i = r.get('inline', {}).get('latency_ms', 0)
        total_latencies.append(s + i)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(total_latencies, bins=20, color='#9b59b6', alpha=0.7, edgecolor='white')
    ax.axvline(lat['avg_total_ms'], color='red', linestyle='--', label=f"Mean: {lat['avg_total_ms']:.0f}ms")
    ax.set_xlabel('Latency (ms)')
    ax.set_ylabel('Count')
    ax.set_title('Latency Distribution per Snippet')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Token Usage Analysis

In [ ]:
tok = metrics['tokens']
cost = metrics['cost_estimate']

print(f"Total Prompt Tokens:     {tok['total_prompt_tokens']:,}")
print(f"Total Completion Tokens: {tok['total_completion_tokens']:,}")
print(f"Total Tokens:            {tok['total_tokens']:,}")
print(f"Avg Tokens/Snippet:      {tok['avg_tokens_per_snippet']:,.0f}")
print()
print(f"Est. Input Cost:   ${cost['input_cost_usd']:.4f}")
print(f"Est. Output Cost:  ${cost['output_cost_usd']:.4f}")
print(f"Est. Total Cost:   ${cost['total_cost_usd']:.4f}")
print(f"Est. Cost/Review:  ${cost['cost_per_review_usd']:.6f}")

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Token breakdown pie
    ax1.pie([tok['total_prompt_tokens'], tok['total_completion_tokens']],
            labels=['Prompt', 'Completion'], autopct='%1.0f%%',
            colors=['#3498db', '#e74c3c'], startangle=90)
    ax1.set_title('Token Distribution')
    
    # Per-snippet token histogram
    snippet_tokens = []
    for r in run_data['results']:
        t = 0
        for phase in ['summary', 'inline']:
            usage = r.get(phase, {}).get('usage') or {}
            t += (usage.get('total_tokens') or 0)
        snippet_tokens.append(t)
    
    ax2.hist(snippet_tokens, bins=20, color='#2ecc71', alpha=0.7, edgecolor='white')
    ax2.set_xlabel('Tokens per Snippet')
    ax2.set_ylabel('Count')
    ax2.set_title('Token Usage Distribution')
    
    plt.tight_layout()
    plt.show()

## 7. Detection Rate by Code Complexity

Does snippet length (as a proxy for complexity) affect detection?

In [ ]:
# Group vulnerable snippets by line count
from evaluation.evaluate import load_dataset

vulnerable_ds, _ = load_dataset()
snippet_map = {s['id']: s for s in vulnerable_ds}

complexity_bins = {'1-3 lines': [], '4-6 lines': [], '7+ lines': []}

for r in run_data['results']:
    if not r['is_vulnerable']:
        continue
    snippet = snippet_map.get(r['snippet_id'])
    if not snippet:
        continue
    line_count = len(snippet['code'].split('\n'))
    detected = (len(r.get('summary', {}).get('found_issues', [])) > 0 or
                len(r.get('inline', {}).get('findings', [])) > 0)
    if line_count <= 3:
        complexity_bins['1-3 lines'].append(detected)
    elif line_count <= 6:
        complexity_bins['4-6 lines'].append(detected)
    else:
        complexity_bins['7+ lines'].append(detected)

print(f"{'Complexity':<15} {'Total':>6} {'Detected':>9} {'Recall':>8}")
print('-' * 40)
for label, detections in complexity_bins.items():
    total = len(detections)
    found = sum(detections)
    recall = found / total if total > 0 else 0
    print(f"{label:<15} {total:>6} {found:>9} {recall:>7.0%}")

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 5))
    labels = list(complexity_bins.keys())
    recalls = [sum(d)/len(d) if d else 0 for d in complexity_bins.values()]
    counts = [len(d) for d in complexity_bins.values()]
    
    bars = ax.bar(labels, recalls, color=['#3498db', '#2ecc71', '#e74c3c'], alpha=0.8)
    for bar, val, cnt in zip(bars, recalls, counts, strict=False):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.0%}\n(n={cnt})', ha='center', fontsize=10)
    ax.set_ylabel('Detection Rate')
    ax.set_title('Detection Rate by Code Complexity (Line Count)')
    ax.set_ylim(0, 1.15)
    plt.tight_layout()
    plt.show()

## 8. Failure Analysis & Recommendations

Identify which snippets were missed (false negatives) to guide prompt improvements.

In [ ]:
missed = []
for r in run_data['results']:
    if not r['is_vulnerable']:
        continue
    summary_issues = r.get('summary', {}).get('found_issues', [])
    inline_findings = r.get('inline', {}).get('findings', [])
    if len(summary_issues) == 0 and len(inline_findings) == 0:
        missed.append(r)

false_pos = []
for r in run_data['results']:
    if r['is_vulnerable']:
        continue
    summary_issues = r.get('summary', {}).get('found_issues', [])
    inline_findings = r.get('inline', {}).get('findings', [])
    if len(summary_issues) > 0 or len(inline_findings) > 0:
        false_pos.append(r)

print(f'=== FALSE NEGATIVES (missed vulnerabilities): {len(missed)} ===')
for r in missed:
    print(f"  {r['snippet_id']:>15} | type={r.get('vulnerability_type', 'N/A'):<25} severity={r.get('expected_severity', 'N/A')}")

print(f'\n=== FALSE POSITIVES (safe code flagged): {len(false_pos)} ===')
for r in false_pos:
    issues = r.get('summary', {}).get('found_issues', []) + r.get('inline', {}).get('findings', [])
    print(f"  {r['snippet_id']:>15} | flagged_issues={len(issues)}")

print('\n=== RECOMMENDATIONS ===')
if missed:
    missed_types = Counter(r.get('vulnerability_type') for r in missed)
    print('\nMost-missed vulnerability types:')
    for vtype, count in missed_types.most_common(5):
        print(f'  - {vtype}: {count} missed')
    print('\nPrompt improvement suggestions:')
    print('  1. Add explicit examples of missed categories to system prompts')
    print('  2. Include CWE references in the review instructions')
    print('  3. Consider separate targeted passes for weak categories')
else:
    print('  All vulnerabilities detected — focus on reducing false positives.')

if false_pos:
    print(f'\n  {len(false_pos)} false positives detected.')
    print('  Consider raising min_inline_comment_confidence threshold.')

## 9. Full Markdown Report

In [ ]:
from IPython.display import Markdown

report = generate_markdown_report(metrics)
Markdown(report)